# backprop-pop-outgrad-loop — ex2: backprop with per-leaf accumulation-count diagnostic

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `backprop-pop-outgrad-loop`. Running the final beacon cell reports progress against the `Backprop: backprop pop-outgrad loop` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: backprop pop-outgrad loop` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backprop-pop-outgrad-loop`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backprop-pop-outgrad-loop"
DD_SUBTOPIC = "Backprop: backprop pop-outgrad loop"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Backprop loop + per-leaf accumulation counts — quick refresher

Same reverse-pass driver, instrumented: alongside the normal `.grad` accumulation, keep a parallel `{id(leaf): count}` of how many gradient contributions each leaf received. Each `+=` increments the count.

```python
counts = {}
for node in sorted_graph:
    grad_out = grads.pop(id(node))
    if node.recipe is None:
        node.grad = grad_out if node.grad is None else node.grad + grad_out
        counts[id(node)] = counts.get(id(node), 0) + 1   # <-- count++
        continue
    for argnum, parent in node.recipe.parents.items():
        back_fn = back_funcs[(node.recipe.func, argnum)]
        grad_parent = back_fn(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
        grads[id(parent)] = grads.get(id(parent), 0) + grad_parent
```

**Use case.** A diamond DAG (z appearing at parents={0: z, 1: z} of `z * z`) should yield `counts[id(z)] == 1` — z is a leaf, it gets ONE merged grad after both paths feed in. A leaf consumed by `k` separate downstream operations gets `counts[id(leaf)] == k`. Counts diagnose where grad routed.

### Exercise 2 — backprop with per-leaf accumulation-count diagnostic

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the reverse-pass driver pattern while instrumenting it with a parallel per-leaf accumulation-count dict, so the caller can verify how many gradient contributions each leaf received.
> Keywords: reverse-pass, diagnostic, accumulation-count, diamond-dag
> ```

**KCs targeted:** `backprop-pop-outgrad-loop`, `dispatch-back-fn-from-recipe`

Implement `backprop_counted(end_node, end_grad, sorted_graph, back_funcs)` — same as the ex1 reverse-pass driver, plus return a `{id(leaf): count}` dict tracking how many `.grad` accumulations each leaf received during this backward pass.

**Signature.**
```python
def backprop_counted(end_node, end_grad, sorted_graph, back_funcs):
    ...
    return counts   # {id(leaf): int}
```

**Semantics.** A leaf node is a `MiniTensor` with `recipe is None`. Each time the reverse pass reaches a leaf in `sorted_graph` and writes/adds to `.grad`, increment `counts[id(leaf)]` by 1. Non-leaves do not appear in `counts`.

**Use case.** In a diamond DAG `out = z * z`, leaf `z` appears as `parents={0: z, 1: z}`. Both arg-0 and arg-1 contribute back into `grads[id(z)]` BEFORE `z` itself is popped — so when `z`'s turn comes, `.grad` is written ONCE with the merged value. Expected `counts[id(z)] == 1`. In contrast, a leaf consumed by k separate downstream operations (each its own node in sorted_graph) would accumulate k times — once per consumer's reverse step.

**Algorithm.** Same as ex1 plus the counter increment:
```python
grads = {id(end_node): end_grad}
counts = {}
for node in sorted_graph:
    if id(node) not in grads: continue
    grad_out = grads.pop(id(node))
    if node.recipe is None:
        node.grad = grad_out if node.grad is None else node.grad + grad_out
        counts[id(node)] = counts.get(id(node), 0) + 1
        continue
    for argnum, parent in node.recipe.parents.items():
        bf = back_funcs[(node.recipe.func, argnum)]
        gp = bf(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
        grads[id(parent)] = grads.get(id(parent), 0) + gp
return counts
```

Same three invariants as ex1: pop-don't-peek, accumulate-don't-overwrite, leaves-get-`.grad`-non-leaves-stay-in-`grads`.

In [ ]:
def backprop_counted(end_node, end_grad, sorted_graph, back_funcs) -> dict:
    grads = {id(end_node): end_grad}
    counts = {}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue
        grad_out = grads.pop(nid)         # POP — node done after this
        if node.recipe is None:
            # Leaf: accumulate (don't overwrite) into .grad + bump count.
            if node.grad is None:
                node.grad = grad_out
            else:
                node.grad = node.grad + grad_out
            counts[nid] = counts.get(nid, 0) + 1
            continue
        # Non-leaf: dispatch + accumulate into each parent's slot.
        for argnum, parent in node.recipe.parents.items():
            back_fn = back_funcs[(node.recipe.func, argnum)]
            grad_parent = back_fn(
                grad_out, node.array,
                *node.recipe.args, **node.recipe.kwargs,
            )
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + grad_parent
    return counts


<details><summary>Solution</summary>

```python
def backprop_counted(end_node, end_grad, sorted_graph, back_funcs) -> dict:
    grads = {id(end_node): end_grad}
    counts = {}
    for node in sorted_graph:
        nid = id(node)
        if nid not in grads:
            continue
        grad_out = grads.pop(nid)         # POP — node done after this
        if node.recipe is None:
            # Leaf: accumulate (don't overwrite) into .grad + bump count.
            if node.grad is None:
                node.grad = grad_out
            else:
                node.grad = node.grad + grad_out
            counts[nid] = counts.get(nid, 0) + 1
            continue
        # Non-leaf: dispatch + accumulate into each parent's slot.
        for argnum, parent in node.recipe.parents.items():
            back_fn = back_funcs[(node.recipe.func, argnum)]
            grad_parent = back_fn(
                grad_out, node.array,
                *node.recipe.args, **node.recipe.kwargs,
            )
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + grad_parent
    return counts
```

**Counts diagnose routing, not values.** Two graphs can produce the same numerical `.grad` via different routings — counts tell you which one actually fired. In a diamond `z*z`, count is 1 (merged in grads dict before leaf pop); in a fork `out1 = z; out2 = z; loss = f(out1, out2)` where the leaf is consumed by TWO separate non-leaf nodes, count is 2.

**Why counts isn't just a length check on grads dict.** Once a leaf is popped, its `id` leaves `grads` — only the cumulative `counts` survives the pass. Useful for off-line analysis of a completed backward.

**Composability with the un-instrumented driver.** Wrapping the existing `backprop` in a counter would require closure tricks; the cleaner approach is to add a counter dict to the ONE function that already knows when the increment should fire.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()